# Tech Challenge Fase 2  
## Notebook 05.1 — Data Quality Bronze

### Responsabilidade do notebook

Este notebook executa as regras de qualidade da camada **Bronze**.

A validação Bronze possui foco técnico e verifica:

- existência das partições;
- presença de arquivos Parquet;
- quantidade mínima de registros;
- disponibilidade do schema;
- presença de colunas obrigatórias;
- presença de metadados técnicos;
- consistência da estrutura por ano;
- geração de resultados detalhados e consolidados.

---

### Entradas

```text
config/config.json
config/quality_metadata
bronze/<dataset>/ano=YYYY
```

### Saídas

```text
logs/data_quality/bronze/details
logs/data_quality/bronze/summary
logs/data_quality/rejected/bronze
logs/data_quality/history/bronze
```

## 1. Contexto da qualidade Bronze

A Bronze preserva os dados o mais próximo possível da origem.

Por isso, as regras aplicadas nesta camada não devem alterar os dados.  
Elas apenas avaliam se a ingestão ocorreu de forma íntegra e rastreável.

```text
Raw
 ↓
Bronze  ← validação técnica
 ↓
Silver
```

## 2. Importação das bibliotecas

Nesta etapa importamos recursos para:

- leitura das configurações;
- leitura da `quality_metadata`;
- inspeção de diretórios;
- validação de schemas;
- execução das regras;
- criação de resultados com schema explícito.

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType
)

## 3. Leitura das configurações oficiais

Os caminhos são carregados exclusivamente do `config.json`, garantindo alinhamento com o setup e com os demais notebooks.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    dbutils.fs.head(CONFIG_FILE_PATH)
)

BRONZE_PATH = config["paths"]["bronze_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

QUALITY_ROOT_PATH = f"{LOG_PATH}/data_quality"
QUALITY_BRONZE_PATH = f"{QUALITY_ROOT_PATH}/bronze"
QUALITY_DETAILS_PATH = f"{QUALITY_BRONZE_PATH}/details"
QUALITY_SUMMARY_PATH = f"{QUALITY_BRONZE_PATH}/summary"
QUALITY_REJECTED_PATH = f"{QUALITY_ROOT_PATH}/rejected/bronze"
QUALITY_HISTORY_PATH = f"{QUALITY_ROOT_PATH}/history/bronze"

print("BRONZE_PATH:", BRONZE_PATH)
print("QUALITY_BRONZE_PATH:", QUALITY_BRONZE_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Criação dos diretórios de saída

Nesta etapa validamos os diretórios utilizados para:

- resultados detalhados;
- resumo por dataset;
- registros inválidos;
- histórico de qualidade.

In [0]:
for path in [
    QUALITY_BRONZE_PATH,
    QUALITY_DETAILS_PATH,
    QUALITY_SUMMARY_PATH,
    QUALITY_REJECTED_PATH,
    QUALITY_HISTORY_PATH
]:
    dbutils.fs.mkdirs(path)

print("Diretórios de qualidade Bronze criados/validados.")

## 5. Leitura das regras Bronze

O notebook consome apenas as regras:

```text
layer = bronze
enabled = true
```

registradas no `05_0_quality_orquestrador`.

In [0]:
quality_metadata_path = (
    f"{CONFIG_PATH}/quality_metadata"
)

df_quality_metadata = (
    spark.read
    .parquet(quality_metadata_path)
)

df_bronze_rules = (
    df_quality_metadata
    .filter(
        (F.col("layer") == "bronze")
        & F.col("enabled")
    )
    .orderBy("dataset", "rule_id")
)

display(df_bronze_rules)

bronze_rules = [
    row.asDict()
    for row in df_bronze_rules.collect()
]

print(
    "Quantidade de regras Bronze:",
    len(bronze_rules)
)

## 6. Definição dos datasets e anos avaliados

Os datasets monitorados são obtidos das regras Bronze.

Os anos avaliados seguem a estrutura oficial do projeto:

```text
2023
2024
2025
```

In [0]:
datasets = sorted(
    list({
        item["dataset"]
        for item in bronze_rules
    })
)

anos = [2023, 2024, 2025]

print("Datasets:", datasets)
print("Anos:", anos)

## 7. Schema dos resultados de qualidade

Cada execução de regra gera um registro com:

- camada;
- dataset;
- ano;
- regra;
- severidade;
- status;
- quantidade de registros avaliados;
- quantidade de registros inválidos;
- percentual inválido;
- tolerância;
- mensagem;
- data de execução.

In [0]:
schema_quality_result = StructType([
    StructField("layer", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("rule_id", StringType(), False),
    StructField("rule_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("column_name", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("blocking", BooleanType(), False),
    StructField("tolerance_percent", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("records_evaluated", LongType(), False),
    StructField("invalid_records", LongType(), False),
    StructField("invalid_percent", DoubleType(), False),
    StructField("message", StringType(), True),
    StructField("evaluated_path", StringType(), True),
    StructField("execution_date", StringType(), False),
    StructField("evaluated_at", StringType(), False)
])

## 8. Funções auxiliares para execução das regras

As funções abaixo implementam as regras Bronze cadastradas no orquestrador.

Tipos suportados nesta etapa:

- `path_exists`;
- `row_count_min`;
- `required_columns`.

In [0]:
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def parquet_files_exist(path: str) -> bool:
    try:
        files = dbutils.fs.ls(path)

        return any(
            item.name.endswith(".parquet")
            for item in files
        )
    except Exception:
        return False


def calculate_status(
    invalid_percent: float,
    tolerance_percent: float
) -> str:
    if invalid_percent <= tolerance_percent:
        return "APROVADO"

    if invalid_percent <= max(
        tolerance_percent * 2,
        tolerance_percent + 1.0
    ):
        return "ATENCAO"

    return "REPROVADO"


def required_columns_list(
    column_name: str
) -> list:
    if not column_name:
        return []

    return [
        item.strip()
        for item in column_name.split(",")
        if item.strip()
    ]

## 9. Execução das regras Bronze

Nesta etapa, cada regra é executada para cada dataset e ano.

Nenhum dado é alterado.  
Apenas os resultados das validações são registrados.

In [0]:
quality_results = []
rejected_partitions = []

for dataset in datasets:
    dataset_rules = [
        rule
        for rule in bronze_rules
        if rule["dataset"] == dataset
    ]

    for ano in anos:
        partition_path = (
            f"{BRONZE_PATH}/"
            f"{dataset}/"
            f"ano={ano}"
        )

        partition_exists = path_exists(
            partition_path
        )

        parquet_exists = (
            parquet_files_exist(
                partition_path
            )
            if partition_exists
            else False
        )

        df_partition = None
        records_evaluated = 0
        partition_columns = []

        if partition_exists and parquet_exists:
            try:
                df_partition = (
                    spark.read
                    .parquet(partition_path)
                )

                records_evaluated = (
                    df_partition.count()
                )

                partition_columns = (
                    df_partition.columns
                )
            except Exception as e:
                rejected_partitions.append({
                    "dataset": dataset,
                    "ano": ano,
                    "partition_path": partition_path,
                    "reason": f"erro_leitura_parquet: {str(e)}",
                    "execution_date": EXECUTION_DATE
                })

        for rule in dataset_rules:
            rule_type = rule["rule_type"]
            invalid_records = 0
            invalid_percent = 0.0
            message = ""

            if rule_type == "path_exists":
                invalid_records = (
                    0
                    if partition_exists
                    and parquet_exists
                    else 1
                )

                records_for_rule = 1

                invalid_percent = (
                    invalid_records
                    / records_for_rule
                    * 100
                )

                if invalid_records == 0:
                    message = (
                        "Partição e arquivo Parquet "
                        "encontrados."
                    )
                else:
                    message = (
                        "Partição inexistente ou "
                        "sem arquivo Parquet."
                    )

            elif rule_type == "row_count_min":
                records_for_rule = max(
                    records_evaluated,
                    1
                )

                invalid_records = (
                    0
                    if records_evaluated > 0
                    else 1
                )

                invalid_percent = (
                    0.0
                    if records_evaluated > 0
                    else 100.0
                )

                if records_evaluated > 0:
                    message = (
                        f"Partição possui "
                        f"{records_evaluated} registros."
                    )
                else:
                    message = (
                        "Partição vazia ou indisponível."
                    )

            elif rule_type == "required_columns":
                expected_columns = (
                    required_columns_list(
                        rule["column_name"]
                    )
                )

                missing_columns = [
                    column
                    for column in expected_columns
                    if column not in partition_columns
                ]

                records_for_rule = max(
                    len(expected_columns),
                    1
                )

                invalid_records = len(
                    missing_columns
                )

                invalid_percent = (
                    invalid_records
                    / records_for_rule
                    * 100
                )

                if missing_columns:
                    message = (
                        "Colunas ausentes: "
                        + ", ".join(
                            missing_columns
                        )
                    )
                else:
                    message = (
                        "Todas as colunas "
                        "obrigatórias estão presentes."
                    )

            else:
                records_for_rule = 1
                invalid_records = 1
                invalid_percent = 100.0
                message = (
                    f"Tipo de regra não suportado "
                    f"na Bronze: {rule_type}"
                )

            status = calculate_status(
                invalid_percent,
                rule["tolerance_percent"]
            )

            quality_results.append({
                "layer": "bronze",
                "dataset": dataset,
                "ano": int(ano),
                "rule_id": rule["rule_id"],
                "rule_name": rule["rule_name"],
                "rule_type": rule_type,
                "column_name": rule["column_name"],
                "severity": rule["severity"],
                "blocking": bool(rule["blocking"]),
                "tolerance_percent": float(
                    rule["tolerance_percent"]
                ),
                "status": status,
                "records_evaluated": int(
                    records_evaluated
                    if rule_type != "required_columns"
                    else records_for_rule
                ),
                "invalid_records": int(
                    invalid_records
                ),
                "invalid_percent": float(
                    round(
                        invalid_percent,
                        4
                    )
                ),
                "message": message,
                "evaluated_path": partition_path,
                "execution_date": str(
                    EXECUTION_DATE
                ),
                "evaluated_at": (
                    datetime.now()
                    .isoformat()
                )
            })

print(
    "Validações executadas:",
    len(quality_results)
)

## 10. Criação do DataFrame de resultados

O schema explícito garante estabilidade mesmo quando nenhuma regra retorna erro.

In [0]:
df_quality_results = spark.createDataFrame(
    quality_results,
    schema=schema_quality_result
)

display(
    df_quality_results
    .orderBy(
        "dataset",
        "ano",
        "rule_id"
    )
)

## 11. Persistência dos resultados detalhados

Os resultados serão armazenados por data de execução para permitir histórico e auditoria.

In [0]:
details_output_path = (
    f"{QUALITY_DETAILS_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_results
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(details_output_path)
)

print("Detalhes salvos em:")
print(details_output_path)

## 12. Criação do resumo por dataset

O resumo apresenta:

- quantidade total de regras;
- regras aprovadas;
- regras em atenção;
- regras reprovadas;
- regras críticas reprovadas;
- score de qualidade.

In [0]:
df_quality_summary = (
    df_quality_results
    .groupBy(
        "layer",
        "dataset",
        "ano"
    )
    .agg(
        F.count("*").alias(
            "total_rules"
        ),
        F.sum(
            F.when(
                F.col("status") == "APROVADO",
                1
            ).otherwise(0)
        ).alias(
            "approved_rules"
        ),
        F.sum(
            F.when(
                F.col("status") == "ATENCAO",
                1
            ).otherwise(0)
        ).alias(
            "warning_rules"
        ),
        F.sum(
            F.when(
                F.col("status") == "REPROVADO",
                1
            ).otherwise(0)
        ).alias(
            "failed_rules"
        ),
        F.sum(
            F.when(
                (F.col("status") == "REPROVADO")
                & (
                    F.col("severity")
                    == "CRITICAL"
                ),
                1
            ).otherwise(0)
        ).alias(
            "critical_failed_rules"
        )
    )
    .withColumn(
        "quality_score_percent",
        F.round(
            (
                F.col("approved_rules")
                / F.col("total_rules")
            ) * 100,
            2
        )
    )
    .withColumn(
        "dataset_status",
        F.when(
            F.col("critical_failed_rules") > 0,
            "REPROVADO"
        )
        .when(
            F.col("failed_rules") > 0,
            "ATENCAO"
        )
        .otherwise("APROVADO")
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_quality_summary
    .orderBy("dataset", "ano")
)

## 13. Persistência do resumo Bronze

Este resultado será consumido pelo dashboard de qualidade.

In [0]:
summary_output_path = (
    f"{QUALITY_SUMMARY_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_quality_summary
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(summary_output_path)
)

print("Resumo salvo em:")
print(summary_output_path)

## 14. Persistência das partições rejeitadas

Caso uma partição não possa ser lida, seu erro é registrado para auditoria.

In [0]:
schema_rejected_partition = StructType([
    StructField("dataset", StringType(), False),
    StructField("ano", IntegerType(), False),
    StructField("partition_path", StringType(), False),
    StructField("reason", StringType(), False),
    StructField("execution_date", StringType(), False)
])

df_rejected_partitions = spark.createDataFrame(
    rejected_partitions,
    schema=schema_rejected_partition
)

rejected_output_path = (
    f"{QUALITY_REJECTED_PATH}/"
    f"execution_date={EXECUTION_DATE}"
)

(
    df_rejected_partitions
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(rejected_output_path)
)

print(
    "Partições rejeitadas:",
    df_rejected_partitions.count()
)

## 15. Histórico de qualidade Bronze

Os resultados detalhados são adicionados ao histórico para análise temporal da qualidade.

In [0]:
history_output_path = (
    f"{QUALITY_HISTORY_PATH}"
)

(
    df_quality_results
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("execution_date")
    .save(history_output_path)
)

print("Histórico atualizado em:")
print(history_output_path)

## 16. Checklist final

O notebook verifica se existem regras críticas reprovadas.

Como este notebook é de auditoria, os resultados são persistidos antes da interrupção.

In [0]:
critical_failures = (
    df_quality_results
    .filter(
        (F.col("status") == "REPROVADO")
        & F.col("blocking")
    )
)

critical_failure_count = (
    critical_failures.count()
)

if critical_failure_count > 0:
    display(
        critical_failures
        .orderBy(
            "dataset",
            "ano",
            "rule_id"
        )
    )

    raise Exception(
        f"Foram encontradas "
        f"{critical_failure_count} "
        f"regras bloqueantes reprovadas "
        f"na camada Bronze."
    )

print("Data Quality Bronze concluída com sucesso.")
print("Nenhuma regra bloqueante foi reprovada.")

## Resultado esperado

Ao final deste notebook estarão disponíveis:

```text
logs/data_quality/bronze/details/execution_date=YYYY-MM-DD
logs/data_quality/bronze/summary/execution_date=YYYY-MM-DD
logs/data_quality/rejected/bronze/execution_date=YYYY-MM-DD
logs/data_quality/history/bronze
```

### Próximo notebook

```text
05_2_quality_silver
```